# XÂY DỰNG MÔ HÌNH KNN REGRESSION


**Dữ liệu:** `train_data.csv` và `test_data.csv` đã chuẩn bị  
**Tham số mô hình:** `K = 13`

## Mục tiêu thực hiện

1. Đọc hai tập dữ liệu Train và Test đã được chuẩn bị.
2. Tách các thuộc tính đầu vào `X` và giá trị cần dự đoán `y`.
3. Xây dựng Pipeline gồm `StandardScaler` và `KNeighborsRegressor`.
4. Huấn luyện mô hình bằng tập Train với `K = 13`.
5. Dự đoán trên tập Test và xuất bảng giá trị thực tế – dự đoán.
6. Lưu mô hình để có thể sử dụng lại.



## 1. Import thư viện

Sử dụng Pandas để đọc dữ liệu, Scikit-learn để xây dựng Pipeline và Joblib để lưu mô hình.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

print("Đã import các thư viện cần thiết.")

Đã import các thư viện cần thiết.


## 2. Tải dữ liệu lên Google Colab

Tải đồng thời hai file `train_data.csv` và `test_data.csv`

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Đã tải lên:", list(uploaded.keys()))

Saving train_data.csv to train_data (1).csv
Saving test_data.csv to test_data (1).csv
Đã tải lên: ['train_data (1).csv', 'test_data (1).csv']


## 3. Đọc và kiểm tra dữ liệu

Bộ dữ liệu dùng 8 thuộc tính đầu vào và cột mục tiêu `MedHouseVal`. Kích thước theo dữ liệu đã bàn giao là 16.512 dòng Train và 4.128 dòng Test.

In [ ]:
from pathlib import Path

train_path = Path("train_data.csv")
test_path = Path("test_data.csv")

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "Cần tải lên đủ hai file train_data.csv và test_data.csv trước khi chạy ô này."
    )

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print("Kích thước tập huấn luyện:", train_data.shape)
print("Kích thước tập kiểm tra:", test_data.shape)

print("\nNăm dòng đầu của tập Train:")
display(train_data.head())

print("\nNăm dòng đầu của tập Test:")
display(test_data.head())

Kích thước tập huấn luyện: (16512, 9)
Kích thước tập kiểm tra: (4128, 9)

Năm dòng đầu của tập Train:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,3.2596,33.0,5.017657,1.006421,2300.0,3.691814,32.71,-117.03,1.030
1,3.8125,49.0,4.473545,1.041005,1314.0,1.738095,33.77,-118.16,3.821
2,4.1563,4.0,5.645833,0.985119,915.0,2.723214,34.66,-120.48,1.726
3,1.9425,36.0,4.002817,1.033803,1418.0,3.994366,32.69,-117.11,0.934
4,3.5542,43.0,6.268421,1.134211,874.0,2.300000,36.78,-119.80,0.965



Năm dòng đầu của tập Test:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,1.6812,25.0,4.192201,1.022284,1392.0,3.877437,36.06,-119.01,0.47700
1,2.5313,30.0,5.039384,1.193493,1565.0,2.679795,35.14,-119.46,0.45800
2,3.4801,52.0,3.977155,1.185877,1310.0,1.360332,37.80,-122.44,5.00001
3,5.7376,17.0,6.163636,1.020202,1705.0,3.444444,34.28,-118.72,2.18600
4,3.7250,34.0,5.492991,1.028037,1063.0,2.483645,36.62,-121.93,2.78000


In [ ]:
target_column = "MedHouseVal"

if target_column not in train_data.columns or target_column not in test_data.columns:
    raise ValueError(f"Cả hai tập dữ liệu phải có cột mục tiêu {target_column}.")

if list(train_data.columns) != list(test_data.columns):
    raise ValueError("Các cột trong tập Train và Test không khớp hoặc không cùng thứ tự.")

if train_data.isnull().sum().sum() > 0 or test_data.isnull().sum().sum() > 0:
    raise ValueError("Dữ liệu còn giá trị thiếu. Cần kiểm tra lại file đã tiền xử lý.")

feature_columns = [col for col in train_data.columns if col != target_column]

if len(feature_columns) != 8:
    raise ValueError(f"Mong đợi 8 thuộc tính đầu vào, nhưng nhận được {len(feature_columns)}.")

print("Số mẫu Train:", len(train_data))
print("Số mẫu Test:", len(test_data))
print("Số thuộc tính đầu vào:", len(feature_columns))
print("Các thuộc tính:", feature_columns)
print("Cột mục tiêu:", target_column)

Số mẫu Train: 16512
Số mẫu Test: 4128
Số thuộc tính đầu vào: 8
Các thuộc tính: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Cột mục tiêu: MedHouseVal


## 4. Tách dữ liệu đầu vào X và đầu ra y

- `X_train`, `X_test`: 8 thuộc tính dùng làm dữ liệu đầu vào.
- `y_train`, `y_test`: giá trị `MedHouseVal` cần dự đoán.

Tập Test được giữ riêng, không dùng để huấn luyện.

In [ ]:
X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_test = test_data[feature_columns]
y_test = test_data[target_column]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (16512, 8)
y_train: (16512,)
X_test: (4128, 8)
y_test: (4128,)


## 5. Xây dựng Pipeline KNN Regression với K = 13

Pipeline gồm hai bước:

1. **StandardScaler:** chuẩn hóa các thuộc tính đầu vào về cùng thang đo để khoảng cách giữa các điểm dữ liệu không bị chi phối quá mức bởi thuộc tính có đơn vị lớn.
2. **KNeighborsRegressor:** thuật toán hồi quy KNN. Tham số `n_neighbors=13` nghĩa là khi dự đoán, thuật toán xét 13 mẫu huấn luyện gần nhất và lấy trung bình giá trị mục tiêu của các láng giềng theo thiết lập mặc định.

In [ ]:
K = 13

model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=K))
])

print("Đã khởi tạo Pipeline KNN Regression.")
print("Giá trị K:", K)
print("Các bước trong Pipeline:", list(model.named_steps.keys()))

Đã khởi tạo Pipeline KNN Regression.
Giá trị K: 13
Các bước trong Pipeline: ['scaler', 'knn']


## 6. Huấn luyện mô hình

Lệnh `fit()` cho Pipeline học tham số chuẩn hóa từ tập Train, sau đó xây dựng mô hình KNN dựa trên dữ liệu Train. Tập Test không được dùng trong bước này.

In [ ]:
model.fit(X_train, y_train)

print("HUẤN LUYỆN KNN REGRESSION THÀNH CÔNG!")
print("Số mẫu dùng để huấn luyện:", len(X_train))
print("Số thuộc tính đầu vào:", X_train.shape[1])
print("Giá trị K:", K)

HUẤN LUYỆN KNN REGRESSION THÀNH CÔNG!
Số mẫu dùng để huấn luyện: 16512
Số thuộc tính đầu vào: 8
Giá trị K: 13


## 7. Dự đoán trên tập Test

Sau khi huấn luyện, dùng `predict()` để dự đoán `MedHouseVal` cho 4.128 mẫu trong tập Test.

In [ ]:
y_pred = model.predict(X_test)

print("Đã dự đoán trên tập Test.")
print("Số giá trị dự đoán:", len(y_pred))

Đã dự đoán trên tập Test.
Số giá trị dự đoán: 4128


## 8. Đối chiếu giá trị thực tế và giá trị dự đoán

Hiển thị 10 mẫu đầu tiên để kiểm tra đầu ra của mô hình.

In [ ]:
prediction_preview = pd.DataFrame({
    "Gia_tri_thuc_te": y_test.iloc[:10].to_numpy(),
    "Gia_tri_du_doan": y_pred[:10]
})

display(prediction_preview)

,Gia_tri_thuc_te,Gia_tri_du_doan
0,0.47700,0.559000
1,0.45800,0.853308
2,5.00001,4.308465
3,2.18600,2.628692
4,2.78000,2.580000
5,1.58700,2.206154
6,1.98200,2.541846
7,1.57500,1.928000
8,3.40000,2.073538
9,4.46600,4.761545


## 9. Xuất toàn bộ kết quả dự đoán

Lưu giá trị thực tế và dự đoán của toàn bộ tập Test thành CSV để gửi cho thành viên phụ trách đánh giá.

In [ ]:
prediction_results = pd.DataFrame({
    "Gia_tri_thuc_te": y_test.to_numpy(),
    "Gia_tri_du_doan": y_pred
})

prediction_results.to_csv("knn_predictions_k13.csv", index=False, encoding="utf-8-sig")
print("Đã lưu file knn_predictions_k13.csv")
display(prediction_results.head())

Đã lưu file knn_predictions_k13.csv


,Gia_tri_thuc_te,Gia_tri_du_doan
0,0.47700,0.559000
1,0.45800,0.853308
2,5.00001,4.308465
3,2.18600,2.628692
4,2.78000,2.580000


## 10. Lưu mô hình đã huấn luyện

Dùng Joblib để lưu toàn bộ Pipeline, bao gồm cả StandardScaler đã được học và KNN Regression đã huấn luyện. Khi nạp lại file này, có thể dự đoán dữ liệu mới mà không cần huấn luyện lại.

In [ ]:
model_filename = "knn_regression_model_k13.pkl"
joblib.dump(model, model_filename)

print(f"Đã lưu mô hình: {model_filename}")

Đã lưu mô hình: knn_regression_model_k13.pkl


## 11. Tải các file kết quả về máy

Tải file mô hình `.pkl` để lưu trữ hoặc tích hợp vào chương trình. File CSV chứa dự đoán trên tập Test có thể gửi cho Nho để tính các chỉ số đánh giá.

In [ ]:
files.download("knn_regression_model_k13.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("knn_predictions_k13.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>